# Lecture 02 Practice - Optimization and Search

This notebook contains exercises for the optimization and search lecture. Answers are encoded next to each exercise. Running an answer cell reveals the answer and explanation.


In [6]:
import numpy as np
import matplotlib.pyplot as plt
import itertools
import math
import random

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=3, suppress=True)

ANSWER_KEY = 73

def reveal_answer(encoded):
    raw = base64.b64decode(encoded)
    text = bytes(byte ^ ANSWER_KEY for byte in raw).decode("utf-8")
    from IPython.display import Markdown, display
    display(Markdown(text))

def tour_length(order, cities):
    order = list(order)
    total = 0.0
    for i in range(len(order)):
        total += np.linalg.norm(cities[order[i]] - cities[order[(i + 1) % len(order)]])
    return total


## Exercise 1 - Classify Optimization Problems

For each problem, decide whether it is mainly discrete optimization, continuous optimization, or could naturally involve both.

1. Finding the best learning rate for a neural network.
2. Finding the shortest route through 12 cities.
3. Choosing course times for a university timetable.
4. Fitting the weights of a linear regression model.
5. Choosing which features to include, then fitting model weights.


Finding the best learning rate for a neural network is continuous problem
The shortest route through 12 cities is discrete optimization.
Choosing course times for a university timetable is discreate optimization
Fitting the weights of a linear regression model is continuous problem
Choosing which features to include, then fitting model weights is Mixed discrete-continuous


## Exercise 2 - Exhaustive Search for a Tiny TSP

Use exhaustive search to find the shortest closed tour for the four cities below. Fix city `0` as the first city to avoid counting the same cycle from different starting points.


In [8]:
cities = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [1.0, 1.0],
    [0.0, 1.0],
])



# Fill in the exhaustive search.
# Hint: iterate over permutations of [1, 2, 3] and prepend 0.

small_cities = cities[:4]
best_order = None
best_length = float("inf")
count = 0

for rest in itertools.permutations(range(1, len(small_cities))):
    candidate = (0,) + rest
    length = tour_length(candidate, small_cities)
    count += 1
    if length < best_length:
        best_length = length
        best_order = candidate

print("evaluated candidates:", count)
print("best order:", best_order)
print("best length:", best_length)



print(best_order, best_length)


evaluated candidates: 6
best order: (0, 1, 2, 3)
best length: 4.0
(0, 1, 2, 3) 4.0


### Encoded answer - Exercise 2


## Exercise 3 - Greedy Nearest Neighbor

Implement nearest-neighbor greedy search for the same cities. Start at city `0`, repeatedly choose the nearest unvisited city, and return the final ordering.


In [13]:
def nearest_neighbor_tour(cities, start=0):
    unvised = set(range(len(cities)))
    order = [start]
    unvised.remove(start)
    while unvised:
        current =order[-1]
        next_city = min(unvised, key=lambda j: np.linalg.norm(cities[current] - cities[j]))
        order.append(next_city)
        unvised.remove(next_city)
    return order 

greedy_order = nearest_neighbor_tour(cities,start=0)
print(nearest_neighbor_tour(cities, start=0))
print("greedy order:", greedy_order)



[0, 1, 2, 3]
greedy order: [0, 1, 2, 3]


### Encoded answer - Exercise 3


## Exercise 4 - Hill Climbing on a One-Dimensional Function

Consider the quality function below. Implement hill climbing for maximization using neighbors `x - step` and `x + step`. Stop when neither neighbor improves the current value.

Then consider the neighborhood definition: how might the result change if the neighbors were not only the two closest points, but a wider interval of candidate points around `x`?


In [15]:
def quality(x):
    return np.sin(3 * x) + 0.2 * x

def hill_climb(start, step=0.05, max_steps=200):
    x = start
    path = [x]

    for _ in range(max_steps):

        candidates = [x - step, x + step]

        # only keep candidates inside [-3, 3]
        candidates = [c for c in candidates if -3 <= c <= 3]

        best = max(candidates, key=quality)

        if quality(best) > quality(x):
            x = best
            path.append(x)
        else:
            break

    return np.array(path)


path = hill_climb(start=-2.0)

print(path[-1], quality(path[-1]))

-1.5499999999999996 0.6880544387588794


## Exercise 5 - Exploration and Exploitation

Briefly explain which behavior dominates in each method:

1. Exhaustive search.
2. Hill climbing.
3. Simulated annealing early in the run.
4. Simulated annealing late in the run.


## Exercise 6 - Simulated Annealing Acceptance Probability

For minimization, compute the probability of accepting a worse move when `delta = 2.0` for temperatures `T = 0.2`, `1.0`, and `5.0`.

Use:

$$p = exp(-delta / T).$$


In [18]:
delta = 2.0
temperatures = [0.2, 1.0, 5.0]

# Compute and print the probabilities.

for T in temperatures:
    print(T,np.exp(((-1)*delta)/T))

0.2 4.5399929762484854e-05
1.0 0.1353352832366127
5.0 0.6703200460356393


### Encoded answer - Exercise 6


In [17]:
reveal_answer('amppDDEsOyogOixpf0NDHSEsaTk7JisoKyAlID0gLDppKDssaSg5OTsmMSAkKD0sJTBpKXlneXl5eX18KWVpKXlneHp8KWVpKCctaSl5Z39+eSlnaQEgLiFpPSwkOSw7KD08OyxpJCgiLDppPSEsaSglLiY7ID0hJGkkPCohaSQmOyxpPiAlJSAnLmk9JmkoKiosOT1pPiY7OixpJCY/LDpnQ0MpKSk5MD0hJidDLyY7aR1pICdpPSwkOSw7KD08Oyw6c0NpaWlpOTsgJz1hHWVpJzlnLDE5YWQtLCU9KGlmaR1gYEMpKSk=')


## Exercise 6

The probabilities are approximately `0.000045`, `0.135`, and `0.670`. High temperature makes the algorithm much more willing to accept worse moves.

```python
for T in temperatures:
    print(T, np.exp(-delta / T))
```

## Exercise 7 - Gradient Descent

For the loss function

$$L(w)=(w-3)^2,$$

the derivative is

$$L'(w)=2(w-3).$$

Starting from `w = -1`, perform 10 gradient descent steps with learning rate `0.2`. Print the path.


In [16]:
def loss(w):
    return (w - 3) ** 2

def grad(w):
    return 2 * (w - 3)

w = -1.0
lr = 0.2
path = [w]

def gradient_descent(start,lr,steps):
    w = start 
    path = [w]
    for _ in range(steps):
        w = w-lr*grad(w)
        path.append(w)
    return np.array(path)

# Update w for 10 steps.
path = gradient_descent(-1.0,0.2,10)


print(path)


[-1.     0.6    1.56   2.136  2.482  2.689  2.813  2.888  2.933  2.96
  2.976]


## Exercise 8 - No Free Lunch

Answer briefly.

1. Why is there no universally best optimization algorithm?
2. What kind of problem structure makes gradient descent appropriate?
3. Why might simulated annealing be preferable to pure hill climbing?


### Encoded answer - Exercise 8
